# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

#### SCORING/RANKING
##### I want to find pages ranked by 'CTR opportunity score'. These are pages that are visible enough to matter (>= 1000 impressions) but have low clicks. We return a score for each page, so editors can review top-ranked results first.
##### We want to calculate a priority score that gives the important pages first, not predict a binary result, so the chosen option RANKING/SCORING, is appropriate.

In [8]:
task_type = "Ranking / Scoring"

task_type

'Ranking / Scoring'

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

#### Varaible(s) that affect - pages with adequate impressions, but a low CTR/Engagement rate
#### CTR is the primary proxy, since its measured in the data, can be improved and audited by editors and it is also visible in 'clicks_90d / impressions_90d', so we can calculate expected CTR using pos and content type then flag pages below expectation - "observed outcome, NOT a defined rule"


In [9]:
target_proxy = "CTR (Click-Through Rate)"
label_source = ("Observed: CTR is computed directly from clicks_90d / impressions_90d. "
    "Not derived from a rule. It's measured, not defined.")
label_source

"Observed: CTR is computed directly from clicks_90d / impressions_90d. Not derived from a rule. It's measured, not defined."

## 3. Success metric

*One metric you can defend. What number means 'good'?*

#### A ranking task's success is measured by a metric depicting quality. 
#### Using Spearman rank correlation between our opportunity score and observed CTR. Secondly a good score should correlate strongly with actual CTR (higher score -> higher CTR). The target is a Spearman correlation >= 0.50 means the ranking is useful for prioritization. 

In [10]:
import pandas as pd
from scipy.stats import spearmanr

df = pd.read_csv("d:/Flyrank/Flyrank-assignments/data/raw/content_refresh_anonymized.csv")

# Compute baseline: a simple opportunity score = CTR itself (naive baseline).
baseline_score = df["ctr"].fillna(0)
correlation, pvalue = spearmanr(baseline_score, df["ctr"])

metric_definition = (
    "Spearman rank correlation between our opportunity score and observed CTR. "
    "A score of 0.50+ means editors should see meaningful CTR differences when "
    "they review the top-ranked pages vs. the bottom-ranked."
)

print(f"Success metric: {metric_definition}")
print(f"Baseline (CTR as score): correlation = {correlation:.3f}, p-value = {pvalue:.2e}")


Success metric: Spearman rank correlation between our opportunity score and observed CTR. A score of 0.50+ means editors should see meaningful CTR differences when they review the top-ranked pages vs. the bottom-ranked.
Baseline (CTR as score): correlation = 1.000, p-value = 0.00e+00


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

#### ONE ROW = ONE CONTENT ITEM (PAGE), Each row represents a unique page with observed metrics over a trailing 90-day window.

In [11]:
import pandas as pd

df = pd.read_csv("d:/Flyrank/Flyrank-assignments/data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print(f"Rows: {len(df)} content items")
print(f"Columns: {df.shape[1]} features/metrics\n")

# pages with meaningful visibility (>= 1000 impressions)
df_opportunity = df[df["impressions_90d"] >= 1000].copy()
print(f"Pages with ≥1000 impressions (our target lane): {len(df_opportunity)}\n")

cols_to_show = ["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position", "engagement_rate", "content_age_days"]
print("Sample rows (5 random pages with ≥1000 impressions):")
print(df_opportunity[cols_to_show].sample(5, random_state=42).to_string())


Dataset shape: (30000, 44)
Rows: 30000 content items
Columns: 44 features/metrics

Pages with ≥1000 impressions (our target lane): 13512

Sample rows (5 random pages with ≥1000 impressions):
                 content_id  impressions_90d  clicks_90d   ctr  avg_position  engagement_rate  content_age_days
28086  content_efcca4291abe             2657           5  0.19          26.0             2.35               287
2496   content_4a381867acbc             1690           0  0.00          12.1             0.00               155
26603  content_c567607d2989             3020           8  0.26          22.8             4.76               537
11260  content_d0d96bb7f622             2435           1  0.04          11.2             0.00                97
8565   content_9029c9f20f02             3841          12  0.31           8.0             0.00               124


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

#### Cannot use Rule based mathod here. 
###### A fixed rule like "CTR < 0.1 → prioritize" fails because:
###### CTR depends on POSITION (pages ranked #1 get easier clicks)
###### CTR depends on CONTENT TYPE (keyword articles vs. comparison articles)
###### CTR depends on INTENT (informational vs. transactional)
###### These factors interact in complex ways.

In [12]:
import pandas as pd

df = pd.read_csv("d:/Flyrank/Flyrank-assignments/data/raw/content_refresh_anonymized.csv")

# pages -> meaningful visibility
df_opp = df[df["impressions_90d"] >= 1000].copy()

print("Why a simple rule fails:\n")

print("1. POSITION EFFECT on CTR:")
print(df_opp.groupby("avg_position").agg({
    "ctr": ["mean", "count"],
    "engagement_rate": "mean"
}).round(3))

print("\n2. CONTENT TYPE effect on CTR:")
print(df_opp.groupby("content_type").agg({
    "ctr": ["mean", "count"],
    "engagement_rate": "mean"
}).round(3))

print("\nReason ML works better:")
print(
    "A model can learn that a page ranked #5 with CTR=0.05 is a better opportunity "
    "to review than a page ranked #50 with CTR=0.02. "
    "A rule cannot easily weigh these competing signals together. "
    "The pattern is real, multi-signal, and shifts by content type and position."
)


Why a simple rule fails:

1. POSITION EFFECT on CTR:
               ctr       engagement_rate
              mean count            mean
avg_position                            
0.2           0.07     1           0.000
0.6           0.07     3          33.333
0.7           0.07     4           5.398
0.8           0.07     2           0.000
0.9           0.06     2           0.000
...            ...   ...             ...
82.6          0.00     1           0.000
83.1          0.01     1          11.110
85.5          0.00     1           0.000
85.8          0.01     1           3.400
88.9          0.00     1           0.000

[628 rows x 3 columns]

2. CONTENT TYPE effect on CTR:
                      ctr        engagement_rate
                     mean  count            mean
content_type                                    
comparison article  0.065     31           0.000
feedly article      0.450     80           2.902
keyword article     0.277  13401           3.176

Reason ML works better

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.